# Word embeddings and sentence embeddings

An embedding is a row of numbers for a word or a text. You try word embeddings and sentence embeddings on a few sentences, then use both to search and plot the AITA replies.

Work with a partner. Fill in the cells that start with a comment and use the check comments to confirm your result. Make a copy named `week09_live_workspace_mine.ipynb`.

In [ ]:
# Install sentence-transformers, gensim, and the other optional NLP packages into the course environment.
# Run this once on your own computer. It can take a few minutes. Skip it on Colab, which already has them.
!uv sync --frozen --group optional-nlp

In [ ]:
import re
import pandas as pd
import gensim.downloader
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import seaborn as sns

pd.set_option("display.max_colwidth", 300)  # show more of each reply

## 1. Word embeddings

In [ ]:
# GloVe: word embeddings trained on Wikipedia and news, with 50 numbers per word.
# The first run downloads about 66 MB.
glove = gensim.downloader.load("glove-wiki-gigaword-50")

In [ ]:
glove["wife"]  # the 50 numbers for "wife"

In [ ]:
glove.most_similar("wife")  # the words with the closest rows. Check: daughter 0.96, husband 0.95

In [ ]:
# Find the closest words to "housework". Check: chores first, 0.79

Are the closest words to *wife* synonyms? What do they have in common?

## 2. From words to sentences

Run these cells and read the scores. Part 3 uses the same code on the AITA replies.

In [ ]:
# One row for a sentence: the average of the rows for its words.
# Stop words and words GloVe does not know are left out.
def average_word_vector(text):
    words = re.findall(r"[a-z]+", text.lower())
    words = [w for w in words if w in glove and w not in ENGLISH_STOP_WORDS]
    return sum(glove[w] for w in words) / len(words)

In [ ]:
query = "You are wrong."
examples = pd.DataFrame({"text": [
    "You are not wrong.",
    "You did nothing wrong.",
    "You are mistaken.",
    "The cat sat on the mat.",
]})

In [ ]:
query_word_row = average_word_vector(query)  # one row of 50 numbers for the query
example_word_rows = [average_word_vector(text) for text in examples["text"]]  # one row for each sentence

In [ ]:
word_scores = cosine_similarity([query_word_row], example_word_rows)  # the [ ] around the query makes it a table with one row
word_scores.round(2)  # one row of scores: the query against each sentence. Check: 1.0, 0.95, 0.71, 0.43

In [ ]:
examples["word_score"] = word_scores[0]  # [0] takes the one row of scores
examples.round(2)

In [ ]:
# A sentence model reads the whole sentence and gives one row of 384 numbers for it.
# The first run downloads about 90 MB.
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
query_sentence_row = model.encode(query)  # one row of 384 numbers for the query
example_sentence_rows = model.encode(examples["text"].tolist())  # one row for each sentence. model.encode needs a list

In [ ]:
sentence_scores = cosine_similarity([query_sentence_row], example_sentence_rows)  # the same steps as for the word averages
examples["sentence_score"] = sentence_scores[0]
examples.round(2)  # Check: 0.83, 0.39, 0.69, 0.01

Which method scores "You are mistaken." above "You did nothing wrong."? Why does the word average give "You are not wrong." a score of 1?

## 3. Search the AITA replies

Use the code from Part 2 on the replies.

In [ ]:
url = "https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/docs/interactives/data/week08/aita_replies.csv"
comments = pd.read_csv(url)
comments["marker"] = comments["body"].str.upper().str.extract(r"^(NTA|YTA)\b")
replies = comments.dropna(subset=["marker"]).reset_index(drop=True)  # keep the replies that start with NTA or YTA
replies["marker"].value_counts()  # Check: NTA 657, YTA 378

In [ ]:
# Make reply_word_rows: one average word vector for each reply in replies["body"]. Check: len(reply_word_rows) is 1035

In [ ]:
# Make reply_sentence_rows with model.encode. This can take a minute. Check: reply_sentence_rows.shape is (1035, 384)

In [ ]:
query = "You should break up with him."

In [ ]:
# Make query_word_row, the average word vector of the query. Check: query_word_row.shape is (50,)

In [ ]:
# Score query_word_row against reply_word_rows and call the result word_scores. Check: word_scores.shape is (1, 1035)

In [ ]:
replies["word_score"] = word_scores[0]  # [0] takes the one row of scores
replies.sort_values("word_score", ascending=False).head(5)[["idstr", "word_score", "body"]]  # Check: the first is t1_einm6sy

In [ ]:
# Make query_sentence_row with model.encode. Check: query_sentence_row.shape is (384,)

In [ ]:
# Score query_sentence_row against reply_sentence_rows and call the result sentence_scores. Check: sentence_scores.shape is (1, 1035)

In [ ]:
# Save sentence_scores[0] in replies["sentence_score"] and show the five highest. Check: the first is t1_ej7bo1p

Which list is about breaking up? Which words of the query are not stop words?

## 4. Plot the replies

PCA turns each row of numbers into two numbers, so each reply can be drawn as a point. The next two cells plot the sentence embeddings.

In [ ]:
pca = PCA(n_components=2)  # keep two numbers per reply
sentence_points = pca.fit_transform(reply_sentence_rows)  # one row of two numbers for each reply
sentence_points.shape  # Check: (1035, 2)

In [ ]:
replies["x"] = sentence_points[:, 0]  # the first number
replies["y"] = sentence_points[:, 1]  # the second number
sns.scatterplot(data=replies, x="x", y="y", hue="marker", s=10)  # each dot is one reply

In [ ]:
# Make word_points from reply_word_rows in the same way. Check: word_points.shape is (1035, 2)

In [ ]:
# Plot word_points in the same way, colored by marker

In which plot are the NTA and YTA replies further apart? Every reply starts with NTA or YTA. Could that explain the plots?

## 5. Your own search

Write a query about another topic in the replies, such as housework or a wedding. Show the five closest replies with each method. Which method finds more replies about your topic?

In [ ]:
# Your query and the five closest replies with each method

## If you have time: find texts like one you already have

The row of any text can be the query. In a project, this finds more texts like one you want to quote or label.

In [ ]:
example_row = reply_sentence_rows[280]  # reply 280: "Nta. Honeatly break up with him. You can do better"
replies["example_score"] = cosine_similarity([example_row], reply_sentence_rows)[0]

In [ ]:
replies.sort_values("example_score", ascending=False).head(6)[["idstr", "example_score", "body"]]  # the first is reply 280 itself. Check: the second is t1_egpmt9p

In [ ]:
# Pick another reply from today's tables (its row number is on the left) and show the five replies closest to it

Save the notebook, restart the kernel, and run every cell.